# Final five-fold OOF evaluation — A2-MP vs D1

This is the final model-selection protocol: fold-local hard-negative mining, full-MP4 outer validation, and one OOF probability per video. Run the cache cells first; do not start two writers for the same cache.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_final_oof_cache import Config as CacheConfig, build_context as build_cache_context, cache_preflight, context_report as cache_report, ensure_feature_cache, write_summary
from v3_final_oof_cv import run_final_oof_cv

RUN_FULL_RGB_CACHE = False
RUN_FULL_MOTION_CACHE = False
RUN_FINAL_CV_AFTER_COMPLETE_CACHE = False


In [ ]:
# 1) Deterministic union and a real-time estimate before the long feature extraction.
cache_context = build_cache_context(CacheConfig())
print(json.dumps(cache_report(cache_context), ensure_ascii=False, indent=2))
preflight = cache_preflight(cache_context)
print(json.dumps(preflight, ensure_ascii=False, indent=2))

In [ ]:
# 2) Resumable cache construction. Enable only when no other writer is active.
rgb_cache = ensure_feature_cache(cache_context, 'rgb', run_full=RUN_FULL_RGB_CACHE)
motion_cache = ensure_feature_cache(cache_context, 'motion', run_full=RUN_FULL_MOTION_CACHE)
print(json.dumps(write_summary(cache_context, rgb_cache, motion_cache, preflight), ensure_ascii=False, indent=2))

In [ ]:
# 3) Run only after both complete caches exist. This performs 5 x fold-local training and OOF statistics.
if RUN_FINAL_CV_AFTER_COMPLETE_CACHE:
    result = run_final_oof_cv()
    print(json.dumps(result['summary'], ensure_ascii=False, indent=2))
    display(result['comparison'])
    display(result['bootstrap'])
else:
    print('Keep this False until RGB and motion caches are complete.')